In [1]:
import os,certifi
from langchain_groq import ChatGroq
from langchain_tavily import TavilySearch
from dotenv import load_dotenv,find_dotenv
from langchain.tools import tool
import requests

load_dotenv()

True

In [2]:
from langchain.agents import create_agent

In [3]:
!uv pip show langchain

Name: langchain
Version: 1.4.7
Location: D:\Downloads\python\ai_agents\agent_1\.venv\Lib\site-packages
Requires: langchain-core, langgraph, pydantic
Required-by: langchain-tavily


In [4]:
find_dotenv()

'd:\\Downloads\\python\\ai_agents\\agent_1\\.env'

In [5]:
search_tool=TavilySearch(max_results=2)

In [29]:
# search_tool.invoke("Give the latest news on CJP?")

In [6]:
url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={os.getenv("weather_api_key")}&query={"abcd"}"
    )

In [7]:
requests.get(url).status_code

400

In [8]:
@tool
def get_wether(city:str)->str:
    """
    Fetch current weather information for a city.
    """

    url=(
        f"https://api.weatherstack.com/current?"
        f"access_key={os.getenv("weather_api_key")}&query={city}"
    )

    res=requests.get(url)
    

    if res.status_code != 200:
        return f"could not fetch wether of {city}"

    data=res.json()

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )





In [10]:
get_wether.invoke("balasore")

'City: balasore\nTemperature: 24°C\nWeather: Light rain shower\nHumidity: 89%'

In [11]:
prompt="""

You are an helpful AI assistant.
Use the avaliable tools to answer the question if nedded.
answer directly if you do not need the tools.

"""

In [12]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=os.getenv("groq_api_key"),
    temperature=0
)

In [24]:
import time

In [14]:
tools=[search_tool,get_wether]

In [15]:
agent=create_agent(
    model=llm,
    tools=tools,
    system_prompt=prompt
    
)

In [22]:
agent.invoke({
    "messages": [
        {"role": "user", "content": "Find the capital of India,and then find its current weather."}
    ]
})['messages'][-1]

AIMessage(content='The capital of India is **New\u202fDelhi**.  \n\nCurrent weather in New\u202fDelhi (as of the latest report):\n- **Temperature:** 24\u202f°C  \n- **Condition:** Patchy rain nearby  \n- **Humidity:** 61%', additional_kwargs={'reasoning_content': 'We need to answer: capital of India and its current weather. We have weather info from tool: City: New Delhi, Temperature 24°C, Weather: Patchy rain nearby, Humidity 61%. Provide answer.'}, response_metadata={'token_usage': {'completion_tokens': 105, 'prompt_tokens': 433, 'total_tokens': 538, 'completion_time': 0.219838033, 'completion_tokens_details': {'reasoning_tokens': 46}, 'prompt_time': 1.3071075699999999, 'prompt_tokens_details': {'cached_tokens': 256}, 'queue_time': 0.21210665, 'total_time': 1.5269456030000002}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_86dc843b12', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a18884-1384-72e2-a3f

In [23]:
url=f"https://v6.exchangerate-api.com/v6/{os.getenv('currency_api_key')}/latest/USD"

In [32]:
requests.get(url).json()['conversion_rates']['USD']

1

In [37]:
@tool
def get_conversion_rate(f:str,t:str,q:int):

    """
    fetch currency conversion rate
    """
    url=f"https://v6.exchangerate-api.com/v6/{os.getenv('currency_api_key')}/latest/USD"

    res=requests.get(url).json()['conversion_rates']

    rate=res[t]/ res[f]

    out= rate*q

    return f"the value of {q}{f} in {t} is {out}"


In [38]:
get_conversion_rate.invoke({"f":"INR","t":"USD","q":100})

'the value of 100INR in USD is 1.0494159475543887'

In [39]:
tools2=[search_tool,get_wether,get_conversion_rate]

In [40]:
agent2=create_agent(
    model=llm,
    tools=tools2,
    system_prompt=prompt

)

In [44]:
agent2.invoke({
    "messages": [
        {"role": "user", "content": "convert 100 AUD into USD."}
    ]
})['messages'][-1].content

'100\u202fAUD is approximately **70.44\u202fUSD** (based on the current conversion rate).'